# Benchmark v2 Kaggle runner

This notebook runs the raw OCR benchmark for one engine per Kaggle session.

Supported engines: `docling`, `hybrid`, `marker`.


In [ ]:
!git clone https://github.com/buinguyenkhai/stock-report-agent-20251.git
%cd stock-report-agent-20251


In [ ]:
!git checkout dev

In [ ]:
!apt-get update -qq
!apt-get install -y -qq tesseract-ocr tesseract-ocr-vie libtesseract-dev libleptonica-dev
%pip install -q pymupdf
%pip install -q -r requirements.txt
%pip install -q "docling[tesserocr]"
%pip install -q marker-pdf surya-ocr


In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["LANGCHAIN_TRACING_V2"] = "false"
os.environ["LANGSMITH_TRACING"] = "false"

DATASET_ROOT = "data/benchmark_v2"
ENGINE = "hybrid"  # docling | hybrid | marker
SPLIT = "dev"
INCLUDE_SCOPE = "included"  # included | all | not_included
DEVICE = "cuda"
HYBRID_THRESHOLD = 0.90
HYBRID_NUMBER_THRESHOLD = 0.95
MARKER_FORCE_OCR = True
MARKER_EXTRACT_IMAGES = False
ENGINE_NAME_MAP = {"docling": "docling", "hybrid": "hybrid_docling", "marker": "marker"}
ENGINE_NAME = ENGINE_NAME_MAP[ENGINE]
RUN_TAG = f"{ENGINE}_{INCLUDE_SCOPE}_{SPLIT}"
PREDICTIONS_ROOT = f"results/benchmark_v2_{RUN_TAG}"
RESULT_JSON = f"results/benchmark_v2_{RUN_TAG}.json"
DEBUG_DIFF_JSON = f"results/benchmark_v2_{RUN_TAG}_debug_diffs.json"

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Engine: {ENGINE}")
print(f"Predictions root: {PREDICTIONS_ROOT}")
print("Benchmark is raw-only; no LLM or structured outputs are used.")


In [ ]:
dataset_root = Path(DATASET_ROOT)
manifest = json.loads((dataset_root / "manifest.json").read_text(encoding="utf-8"))
include_registry = json.loads((dataset_root / "included_samples.json").read_text(encoding="utf-8"))
included_ids = include_registry.get("included_sample_ids", [])
sample_map = {row["sample_id"]: row for row in manifest.get("samples", [])}

if INCLUDE_SCOPE == "included":
    selected_ids = included_ids
elif INCLUDE_SCOPE == "all":
    selected_ids = list(sample_map)
elif INCLUDE_SCOPE == "not_included":
    included_id_set = set(included_ids)
    selected_ids = [sid for sid in sample_map if sid not in included_id_set]
else:
    raise SystemExit(f"Unsupported INCLUDE_SCOPE={INCLUDE_SCOPE}")

missing_artifacts = []
for sid in selected_ids:
    row = sample_map[sid]
    gt_markdown_path = row.get("gt_markdown_path")
    page_image_path = row.get("page_image_path")
    required = []
    if gt_markdown_path:
        required.append(gt_markdown_path)
    else:
        missing_artifacts.append((sid, "<missing gt_markdown_path in manifest>"))
    if page_image_path:
        required.append(page_image_path)
    else:
        missing_artifacts.append((sid, "<missing page_image_path in manifest>"))
    if ENGINE == "marker":
        if row.get("source_pdf_path"):
            required.append(row["source_pdf_path"])
        else:
            missing_artifacts.append((sid, "<missing source_pdf_path in manifest>"))
    for rel in required:
        if not (dataset_root / rel).exists():
            missing_artifacts.append((sid, rel))

if missing_artifacts:
    for sid, rel in missing_artifacts:
        print(f" - {sid}: {rel}")
    raise SystemExit("Dataset validation failed")
print("Dataset validation passed")


In [ ]:
predict_cmd = [sys.executable, "-m", "evaluation.benchmark_v2.predict", "--dataset-root", DATASET_ROOT, "--output-root", PREDICTIONS_ROOT, "--engine", ENGINE, "--split", SPLIT, "--include-scope", INCLUDE_SCOPE, "--device", DEVICE]
if ENGINE == "hybrid":
    predict_cmd.extend(["--hybrid-threshold", str(HYBRID_THRESHOLD), "--hybrid-number-threshold", str(HYBRID_NUMBER_THRESHOLD)])
if ENGINE == "marker":
    if not MARKER_FORCE_OCR:
        predict_cmd.append("--marker-no-force-ocr")
    if MARKER_EXTRACT_IMAGES:
        predict_cmd.append("--marker-extract-images")
print('Running:', ' '.join(predict_cmd))
subprocess.run(predict_cmd, check=True)


In [ ]:
run_cmd = [sys.executable, "-m", "evaluation.benchmark_v2.run", "--dataset-root", DATASET_ROOT, "--predictions-root", PREDICTIONS_ROOT, "--engine-name", ENGINE_NAME, "--split", SPLIT, "--include-scope", INCLUDE_SCOPE, "--output", RESULT_JSON]
print('Running:', ' '.join(run_cmd))
subprocess.run(run_cmd, check=True)
result = json.loads(Path(RESULT_JSON).read_text(encoding="utf-8"))
print(json.dumps(result["summary"], ensure_ascii=False, indent=2))


In [ ]:
diff_cmd = [sys.executable, "-m", "evaluation.benchmark_v2.debug_diffs", "--dataset-root", DATASET_ROOT, "--predictions-root", PREDICTIONS_ROOT, "--split", SPLIT, "--include-scope", INCLUDE_SCOPE, "--output", DEBUG_DIFF_JSON]
subprocess.run(diff_cmd, check=True)
print(Path(DEBUG_DIFF_JSON).resolve())


In [ ]:
import shutil
import zipfile

archive_path = Path(f"results/{RUN_TAG}_artifacts.zip")
archive_path.parent.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for rel in [Path(RESULT_JSON), Path(DEBUG_DIFF_JSON)]:
        if rel.exists():
            zf.write(rel, arcname=rel.as_posix())
    pred_root = Path(PREDICTIONS_ROOT)
    if pred_root.exists():
        for path in pred_root.rglob("*"):
            if path.is_file():
                zf.write(path, arcname=path.as_posix())
print(archive_path.resolve())
